# Science Answer Benchmark — Unseen Question Evaluation
**Research question:** Does science-domain fine-tuning improve classification on entirely new questions, and what happens to the rate at which contradictory answers are treated as correct?

Compare the frozen zero-shot MiniLM and the previously selected fine-tuned checkpoint on the same official SciEntsBank UQ answers. Keep the calibration decision fixed: all answers require review. Use the notebook connected to the training runtime or point it to a saved copy of the checkpoint. No training takes place here.

## 1. Locate the checkpoint
Change the path below if the saved checkpoint is on Drive. Mount Drive first if needed.

In [ ]:
from pathlib import Path

CHECKPOINT = Path("/content/science_nli_finetuned")
# To restore from Drive:
# from google.colab import drive
# drive.mount("/content/drive")
# CHECKPOINT = Path("/content/drive/MyDrive/science_nli_finetuned")

required = ["model.safetensors", "config.json"]
missing = [name for name in required if not (CHECKPOINT / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {CHECKPOINT}. Reconnect to the original training runtime "
        "or restore the saved checkpoint folder, including tokenizer files."
    )
print(f"Found checkpoint: {CHECKPOINT}")


## 2. Load and audit the official UQ split
Check both question IDs and normalized inputs against all three internal splits before inference.

In [ ]:
import hashlib
import json
from pathlib import Path
from urllib.request import urlopen

import pandas as pd

GIT_REV = "ed019d5fd32292198ccd400e03da1248a5174201"
BASE = f"https://raw.githubusercontent.com/Junghwan0516/science-answer-benchmark/{GIT_REV}"
INPUTS = {
    "data/processed/test-unseen-questions.csv": "88cf24e14d5e4af02ed741fc342617b62d8746ef49df492a581c2522c7af3111",
    "data/splits/v1/answer_assignments.csv": "ee1312fe017bb559fdda54e7b31846e1b89db3797abfb6ae3a0b7c20a53da5fa",
}
cache = Path("science-nli-inputs")
cache.mkdir(exist_ok=True)
def verified_csv(remote_path, columns):
    local = cache / Path(remote_path).name
    if not local.exists():
        with urlopen(f"{BASE}/{remote_path}", timeout=60) as response:
            content = response.read()
        if hashlib.sha256(content).hexdigest() != INPUTS[remote_path]:
            raise ValueError(f"Downloaded input differs from pinned data: {remote_path}")
        local.write_bytes(content)
    if hashlib.sha256(local.read_bytes()).hexdigest() != INPUTS[remote_path]:
        raise ValueError(f"Cached input differs from pinned data: {local}")
    return pd.read_csv(local, usecols=columns, keep_default_na=False)

keys = ["sample_id", "question_id", "label", "normalized_input_hash"]
assign = verified_csv("data/splits/v1/answer_assignments.csv", keys + ["internal_split"])
TARGET = {"correct": "entailment", "contradictory": "contradiction",
          "partially_correct_incomplete": "neutral", "irrelevant": "neutral", "non_domain": "neutral"}

uq = verified_csv("data/processed/test-unseen-questions.csv",
                  keys + ["official_split", "question_text", "reference_answer", "student_answer"])
assert len(uq) == 733 and uq.question_id.nunique() == 15 and uq.sample_id.is_unique
assert set(uq.official_split) == {"test-unseen-questions"}
assert assign.internal_split.value_counts().to_dict() == {
    "fit": 3536, "development": 715, "calibration": 718
}
uq["gold"] = uq.label.map(TARGET)
assert uq.gold.value_counts().to_dict() == {"neutral": 368, "entailment": 301, "contradiction": 64}
audit_rows = []
for split, rows in assign.groupby("internal_split"):
    audit_rows.append({
        "internal_split": split,
        "question_id_overlap": len(set(uq.question_id) & set(rows.question_id)),
        "normalized_input_overlap": len(set(uq.normalized_input_hash) & set(rows.normalized_input_hash)),
    })
assert all(row["question_id_overlap"] == row["normalized_input_overlap"] == 0 for row in audit_rows)
display(pd.DataFrame(audit_rows))
display(uq.groupby(["gold", "label"]).size().rename("answers").to_frame())


## 3. Run both models with the same input format
The premise contains the question and reference answer; the student answer is the hypothesis. The predicted class is the largest softmax score. These scores are not calibrated probabilities.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.57.6", "safetensors"], check=True)


In [ ]:
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL = "cross-encoder/nli-MiniLM2-L6-H768"
MODEL_REV = "b95119ce93d3e065de6214e38cd4a97b0f2f2c6d"
LABELS = ["contradiction", "entailment", "neutral"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(2)

@torch.inference_mode()
def predict(records, model, tokenizer, label_names, batch_size=16):
    results = []
    for start in range(0, len(records), batch_size):
        batch = records[start:start+batch_size]
        premise = [f"Question: {r['question']}\nReference answer: {r['reference']}" for r in batch]
        answer = [r["answer"] for r in batch]
        tokens = tokenizer(premise, answer, padding=True, truncation=True,
                           max_length=256, return_tensors="pt").to(device)
        probabilities = model(**tokens).logits.float().softmax(dim=-1).cpu().tolist()
        for row, p in zip(batch, probabilities, strict=True):
            guess = max(range(len(p)), key=p.__getitem__)
            results.append({"id": row["id"], "gold": row["gold"], "prediction": label_names[guess],
                            "scores": {label_names[i]: value for i, value in enumerate(p)}})
    return results



records = [{"id": r.sample_id, "question": r.question_text,
            "reference": r.reference_answer, "answer": r.student_answer, "gold": r.gold}
           for r in uq.itertuples(index=False)]

def run_model(source, **options):
    tokenizer = AutoTokenizer.from_pretrained(source, **options)
    model = AutoModelForSequenceClassification.from_pretrained(
        source, use_safetensors=True, **options
    ).to(device).eval()
    label_names = {int(k): v.lower() for k, v in model.config.id2label.items()}
    assert set(label_names.values()) == set(LABELS)
    return pd.DataFrame(predict(records, model, tokenizer, label_names))

predictions = {}
for name, source, options in [
    ("fine_tuned", CHECKPOINT, {"local_files_only": True}),
    ("zero_shot", MODEL, {"revision": MODEL_REV}),
]:
    predictions[name] = run_model(source, **options)
    print(f"{name}: {len(predictions[name])} predictions on {device}")


## 4. Compare metrics and inspect errors
Correct-answer recall uses all gold entailments as its denominator. Contradiction-to-entailment rate uses all gold contradictions. The frozen review policy is reported separately and has no estimated precision because it accepts nothing.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score

def summarize(frame):
    gold, pred = frame.gold, frame.prediction
    cm = confusion_matrix(gold, pred, labels=LABELS)
    return {
        "answers": len(frame),
        "macro_f1": f1_score(gold, pred, labels=LABELS, average="macro", zero_division=0),
        "correct_recalled": int(cm[1, 1]),
        "gold_correct": int(cm[1].sum()),
        "correct_recall": float(cm[1, 1] / cm[1].sum()),
        "contradictions_marked_correct": int(cm[0, 1]),
        "gold_contradictions": int(cm[0].sum()),
        "contradiction_to_entailment_rate": float(cm[0, 1] / cm[0].sum()),
        "confusion_matrix": cm.tolist(),
    }

results = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
for ax, name in zip(axes, ["zero_shot", "fine_tuned"], strict=True):
    frame = predictions[name]
    assert frame.id.tolist() == uq.sample_id.tolist()
    assert frame.gold.tolist() == uq.gold.tolist()
    results[name] = summarize(frame)
    ConfusionMatrixDisplay(confusion_matrix(frame.gold, frame.prediction, labels=LABELS), display_labels=LABELS).plot(
        ax=ax, colorbar=False, cmap="Blues", xticks_rotation=25
    )
    ax.set_title(name.replace("_", " ").title())
display(pd.DataFrame(results).T.drop(columns="confusion_matrix"))

paired = uq[["sample_id", "question_id", "label", "gold", "question_text",
             "reference_answer", "student_answer"]].copy()
for name, frame in predictions.items():
    paired[name] = frame.prediction.to_numpy()
    paired[f"{name}_entailment_score"] = frame.scores.map(lambda s: s["entailment"]).to_numpy()
paired["review_decision"] = "Needs review"
contradictions = paired.loc[paired.gold.eq("contradiction")]
display(pd.crosstab(contradictions.zero_shot, contradictions.fine_tuned).reindex(
    index=LABELS, columns=LABELS, fill_value=0
))
display(contradictions.loc[contradictions.fine_tuned.eq("entailment")]
        .sort_values("fine_tuned_entailment_score", ascending=False).head(8))

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

report = {
    "dataset_revision": GIT_REV, "split": "test-unseen-questions", "questions": 15,
    "split_audit": audit_rows, "gold_counts": uq.gold.value_counts().to_dict(),
    "base_model": MODEL, "base_revision": MODEL_REV, "labels": LABELS,
    "checkpoint_weights_sha256": sha256_file(CHECKPOINT / "model.safetensors"),
    "torch_version": str(torch.__version__), "transformers_version": transformers.__version__,
    "device": str(device), "max_length": 256, "metrics": results,
    "policy": {"decision": "Needs review", "auto_accepted": 0, "coverage": 0.0,
               "auto_accept_precision": None, "review_required": len(uq)},
}
out = Path("science_nli_uq_results")
out.mkdir(exist_ok=True)
(out / "uq_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
paired.to_csv(out / "uq_paired_predictions.csv", index=False)
fig.savefig(out / "uq_confusion_matrices.png", dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved reports to {out.resolve()}")


## 5. Interpret the comparison
- Did macro F1 and correct-answer recall improve on UQ?
- Did the count of contradictions labeled entailment rise or fall? Which individual answers changed in each direction?
- Inspect a case where the model improved and a case where it still failed. Explain the mismatch using the question and reference answer.
- Report these as benchmark results on 15 questions. Keep UQ separate from threshold selection or future model tuning.